# Adaptive computation of vector leaky modes in a Bragg fiber

**[Jay Gopalakrishnan](https://web.pdx.edu/~gjay/)**

*NGSolve User Meeting, Zürich, June 2026*

<hr>

This notebook demonstrates the adaptive FEM workflow for computing leaky vector (Maxwell) eigenmodes of a Bragg fiber. The adaptive loop follows the classical **Solve → Estimate → Mark → Refine** cycle, using a dual-weighted residual (DWR) error estimator to drive mesh refinement.

At each iteration a Maxwell eigenproblem for leaky modes (each with three electric field components) is solved on the current mesh using nonselfadjoint FEAST, a  contour integral eigensolver;  it obtains sparse matrices from a mixed Nedelec-Lagrange finite element discretization using a  C² smooth PML; the DWR estimator then identifies which elements contribute most to the eigenvalue error, and only those are refined.

The method and construction of error estimators are described in detail in the paper [[1]](#references). The Python package `fibermode` used here is available in GitHub [[3]](#references) and is built atop [NGSolve](https://ngsolve.org)'s Python API.

In [ ]:
import numpy as np
import ngsolve as ng
from ngsolve.webgui import Draw
from scipy.optimize import newton

from fibermode.bragg import BraggExactVector, Bragg

## Fiber geometry

The fiber consists of three concentric regions — an air core, a glass ring, and an air cladding — surrounded by a PML absorbing layer. All refractive indices are specified at the operating wavelength $\lambda = 2.45\,\mu\text{m}$.

In [ ]:
ts    = [4.0775e-05, 1e-5, 1e-5]       # layer thicknesses (m): core | glass | air
mats  = ['air', 'glass', 'air']
ns    = [1.00027717, 1.4388164768221814, 1.00027717]   # refractive indices
wl    = 2.45e-6                        # wavelength (m)
scale = 15e-6                          # characteristic length L (m) for nondimensionalization
maxhs = [.1, .1, .1]                   # initial mesh sizes (fraction of layer radius)

# PML additions appending an absorbing outer layer
ts_pml   = ts   + [5e-5]
mats_pml = mats + ['Outer']
ns_pml   = ns   + [ns[0]]
maxhs_pml = maxhs + [.1]

## Numerical setup

### Simple Bragg geometry 

The `fibermode` package's `Bragg` class builds the NGSolve mesh on the truncated domain including the smooth PML layer. The refractive index is displayed to confirm the geometry.

In [ ]:
bragg_n = Bragg(ts=ts_pml, scale=scale, mats=mats_pml, maxhs=maxhs_pml, ns=ns_pml, wl=wl)

Draw(bragg_n.index, bragg_n.mesh, 'Refractive index');

### Search region in the $Z^2$-plane

Our adaptive eigensolver searches in the nondimensional complex $Z^2$-plane, which is related to the complex plane of the physical propagation constant $\beta$ by

$$
  Z^2 = L^2\bigl(k_0^2 n_0^2 - \beta^2\bigr).
$$

Here $k_0$ is the operating wavenumber (determined by the wavelength `wl` set above) and $n_0$ is the refractive index at infinity (which is that of air, as set above in `ns[2]`).  We initialize the search disk of the  contour integral eigensolver at a $Z^2$ value, where we expect to get a known propagation constant of the fundamental mode. (We know this through semi-analytical computations indicated later, where we compute the numerical error.) 

In [ ]:
# Disk in the Z² plane to search for eigenvalues:  
# (The exact value computed afterward semianalytically is within this disk.)

center = 0.78
radius = 0.1

## Adaptive loop

Each iteration performs:

1. **Solve** — assemble the smooth-PML Maxwell system and run FEAST inside the $Z^2$ contour.
2. **Estimate** — compute the DWR error estimator $\eta_T$ element-by-element.
3. **Mark** — flag elements where $\eta_T > \theta \cdot \max_T \eta_T$ (Dörfler marking, $\theta = 0.1$).
4. **Refine** — bisect marked elements and re-curve the mesh.

These substeps that form one iteration of the adaptivity loop are implemented in a Python generator `leakyvecmodes_adapt_gen` which yields the **Estimator**. Here we  use the generator, instead of the automatic adaptivity loop implemented in `leakyvecmodes_adapt` in order to inspect the results before the mesh is refined again iteratively. Both functions are available as methods of a base `ModeSolver` class from which the `Bragg` class is derived. 

One of the key NGSolve facilities we are using here is `autoupdate=True` argument passed to NGSolve for automatic extensions (of finite element spaces, memory allocation resizings,  and prolongation of grid function data) upon each mesh refinement.

In [ ]:
# Initialize the stepper / generator for the adaptive loop. 

stepper = bragg_n.leakyvecmodes_adapt_gen(
    p=3,
    radius=radius,
    center=center,
    alpha=2,
    maxndofs=200000,
    autoupdate=True,
    verbose=False,
    npts=4,
    nspan=4,
    niterations=100,
    nrestarts=0)

This generator yields a *dictionary* containing the *current state* of the adaptive iterations, including the  current computed eigenvalue and eigenfunction approximations, and the DWR error estimators.

### Iteration 1

In [ ]:
state = next(stepper)

# Plot intensity of electric field & the DWR error estimator on the current mesh:

Draw(ng.Norm(state['uR'].gridfun(i=0).components[0])**2, bragg_n.mesh);
Draw(state['eevis']);  # plot the DWR error estimator on the current mesh

- The eigensolver produced two distinct but close-by eigenvalues (both printed out above). We will see as the iteration proceeds that they appear to merge into a near-degenerate pair representing  the two polarizations of the HE₁₁ mode.
- The computed (right) eigenfunctions corresponding to these two eigenvalues, stored in `state['uR']`, are returned in an object that wraps an NGSolve `MultiVector` object  with further facilities for FEAST iterations.
- From this *"multieigenfunction"* object, the `i`th eigenmode can be made into a `GridFunction` by  `state['uR'].gridfun(i)`. Here it gives the two eigenmodes (each containing three electric field components) for `i=0` and `i=1`. For each `i`, the grid function `state['uR'].gridfun(i)` has two components:
  - the first gives the transverse electric field in the Nedelec space (`state['uR'].gridfun(i).components[0]`)
  - the second gives the scaled longitudinal electric field component in the Lagrange space (`state['uR'].gridfun(i).components[1]`)
     
- Only the intensity of the transverse electric field of the first (`i=0`) mode is plotted above, as second looks similar (as you can verify by changing `i=0` to `i=1` in the above code).

- Perhaps the most interesting plot is the DWR error estimator (plotted on the same mesh). It seems to indicate that the mesh is too coarse in the high-index glass layer. This may be surprising since the intensity of transverse electric field is almost negligible there. But this  finding is consistent with the findings of [[1, 2]](#references).
  

### Iteration 2

In [ ]:
state = next(stepper)

# Plot intensity of electric field & the DWR error estimator on the current mesh:

Draw(ng.Norm(state['uR'].gridfun(i=0).components[0])**2, bragg_n.mesh);
Draw(state['eevis']);  # plot the DWR error estimator on the current mesh

### Iteration 3

In [ ]:
state = next(stepper)

# Plot intensity of electric field &  DWR error estimator on the current mesh:

Draw(ng.Norm(state['uR'].gridfun(i=0).components[0])**2, bragg_n.mesh);
Draw(state['eevis']);  # plot the DWR error estimator on the current mesh

## The reason for the refinements

Taking apart the longitudinal component and studying it closely, we find fine scale oscillations in the glass layer. This is what the DWR estimator was trying to resolve by allocating finer meshes in that region. These fine scale features are easy to miss if we only at the total intensity of the electric field.

In [ ]:
Draw(ng.Norm(state['uR'].gridfun(i=0).components[1])**2, bragg_n.mesh, 
     settings={"Objects": {"Wireframe": False}, "Colormap":{"autoscale": True, "ncolors": 16}});

The importance of capturing these fine scale oscillations when computing confinement losses were clearly shown in the paper [[2]](#references).

## Exact eigenvalue and numerical errors

We use `fibermode`'s  semianalytical facilities, contained in its `BraggExactVector` class, for computing the exact propagation constant $\beta$ of the leaky mode corresponding to the fundamental mode. Then we map this $\beta$ to the non-dimensional exact $Z^2$ eigenvalue and compare it with what the adaptive iterations above produced. 

In [ ]:
bragg_e = BraggExactVector(ts=ts, scale=scale, mats=mats, ns=ns, wl=wl)

nu    = 1      # azimuthal mode number for vector fundamental mode (HE₁₁)
outer = 'h1'   # outgoing solution is the Hankel function of the first kind
k_low = bragg_e.k0 * bragg_e.ns[0] * bragg_e.scale   # lower scaled wavenumber bound

beta_exact = newton(bragg_e.determinant, np.array(.9999 * k_low),
                    args=(nu, outer), tol=1e-15)

print(f'Exact β (scaled)    = {beta_exact}')
print(f'Exact β physical    = {beta_exact/bragg_e.scale}')
print(f'|det| residual      = {abs(bragg_e.determinant(beta_exact, nu, outer)):.2e}')

exact_z2 = bragg_n.sqrZfrom(beta_exact / bragg_n.L)
print(f'Exact Z² = {exact_z2}')

Note that solving for the zero of the transfer-matrix determinant can be rather ill-conditioned, 
as seen by the determinant residual at the computed root, which is only `1.25e-10`. Yet the result we obtained is sufficient for studying convergence as our discretization errors are 
larger. The table below shows the $Z^2$ error against the exact transfer-matrix value and the DWR estimator at each refinement level.

In [ ]:
Zsqrs        = state['Zsqrs']
errestimates = state['errestimates']
ndofs        = state['ndofs']

print(f'{"ndofs":>10}  {"Z² error":>14}  {"DWR estimator":>14}')
for n, z, (eta, _) in zip(ndofs[1:], Zsqrs, errestimates):
    err = abs(z[0] - exact_z2)
    print(f'{n:10d}  {err:14.6e}  {eta:14.6e}')

beta_num  = bragg_n.betafrom(Zsqrs[-1])[0]
beta_exact_phys = beta_exact / bragg_e.scale
print(f'\nNumerical β = {beta_num:.6e} m⁻¹')
print(f'Exact     β = {beta_exact_phys:.6e} m⁻¹')
print(f'|Δβ| / |β|  = {abs(beta_num - beta_exact_phys) / abs(beta_exact_phys):.2e}')

## References

[1] J. Gopalakrishnan, J. Grosek, G. Pinochet-Soto, and P. Vandenberge, "Adaptive resolution of fine scales in modes of microstructured optical fibers," *SIAM Journal on Scientific Computing*, 2025. DOI: [10.1137/24M1651605](https://doi.org/10.1137/24M1651605)

[2] P. Vandenberge, J. Gopalakrishnan, J. Grosek, "Sensitivity of confinement losses in optical fibers to modeling approach," *Optics Express*, 2023. DOI: [10.1364/OE.495467](https://doi.org/10.1364/OE.495467)

[3] `fibermode`, Software hosted at GitHub, Available publicly at  [github.com/jayggg/fibermode](https://github.com/jayggg/fibermode). Code and documentation in development. 2025--.